In [1]:
from kafka import KafkaProducer
import csv
import json
import time
import logging
import os
from datetime import datetime

# Crear carpeta de logs si no existe
DIRECTORIO_LOGS = '../transporte/logs_kafka/'
os.makedirs(DIRECTORIO_LOGS, exist_ok=True)

# Ruta base de los CSV
RUTA_BASE = '../ingesta/datasets_kaggle/'

# Configuración de logs
archivo_log = os.path.join(DIRECTORIO_LOGS, 'kafka_productor.log')
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler(archivo_log, encoding='utf-8')]
)

# Función auxiliar para mostrar en consola y guardar en el log
def log_and_print(mensaje):
    print(mensaje)
    logging.info(mensaje)

# Configuración del productor de Kafka
productor = KafkaProducer(
    bootstrap_servers='192.168.41.51:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    max_block_ms=300000,
    max_request_size=10485760,
    buffer_memory=67108864,
    linger_ms=100,
    retries=5
)

def producir_csv_a_kafka(nombre_archivo, nombre_topico, tamano_lote=10000):
    ruta_csv = os.path.join(RUTA_BASE, nombre_archivo)
    contador = 0
    inicio = datetime.now()

    # Verificar si el archivo existe
    if not os.path.exists(ruta_csv):
        log_and_print(f"Archivo no encontrado: {ruta_csv}")
        logging.error(f"No se pudo encontrar el archivo {ruta_csv}")
        return  # No producimos nada ni creamos topic

    log_and_print(f"Iniciando producción del tópico '{nombre_topico}' desde el archivo '{nombre_archivo}'")

    try:
        with open(ruta_csv, newline='', encoding='utf-8') as archivo_csv:
            lector = csv.DictReader(archivo_csv)

            for i, fila in enumerate(lector, start=1):
                try:
                    productor.send(nombre_topico, fila)
                    contador += 1
                except Exception as error:
                    logging.error(f"Error enviando fila {i} del archivo {nombre_archivo}: {error}")
                    continue

                if i % tamano_lote == 0:
                    productor.flush()
                    log_and_print(f"{i} registros producidos al tópico '{nombre_topico}'")
                    time.sleep(1)

        productor.flush()
        fin = datetime.now()
        duracion = (fin - inicio).total_seconds()
        log_and_print(f"Producción completada del tópico '{nombre_topico}' ({contador} registros en {duracion:.2f} segundos)")

    except Exception as error:
        log_and_print(f"Error al producir el archivo '{nombre_archivo}' al tópico '{nombre_topico}': {error}")
        logging.error(f"Error detallado: {error}")

# Funciones específicas por CSV
def producir_personas():
    producir_csv_a_kafka('Personas.csv', 'Personas')

def producir_coches_ventas():
    producir_csv_a_kafka('CochesVentas.csv', 'CochesVentas')

def producir_info_marcas():
    producir_csv_a_kafka('InfoMarcas.csv', 'InfoMarcas')

def producir_modelos_coches():
    producir_csv_a_kafka('ModelosCoches.csv', 'ModelosCoches')

# Main
if __name__ == "__main__":
    producir_info_marcas()
    producir_personas()
    producir_modelos_coches()
    producir_coches_ventas()

    productor.close()
    log_and_print("Todos los tópicos fueron producidos correctamente.")

Iniciando producción del tópico 'InfoMarcas' desde el archivo 'InfoMarcas.csv'
Producción completada del tópico 'InfoMarcas' (40 registros en 0.41 segundos)
Iniciando producción del tópico 'Personas' desde el archivo 'Personas.csv'
10000 registros producidos al tópico 'Personas'
20000 registros producidos al tópico 'Personas'
Producción completada del tópico 'Personas' (23906 registros en 10.18 segundos)
Iniciando producción del tópico 'ModelosCoches' desde el archivo 'ModelosCoches.csv'
10000 registros producidos al tópico 'ModelosCoches'
20000 registros producidos al tópico 'ModelosCoches'
30000 registros producidos al tópico 'ModelosCoches'
40000 registros producidos al tópico 'ModelosCoches'
Producción completada del tópico 'ModelosCoches' (47523 registros en 89.29 segundos)
Iniciando producción del tópico 'CochesVentas' desde el archivo 'CochesVentas.csv'
10000 registros producidos al tópico 'CochesVentas'
20000 registros producidos al tópico 'CochesVentas'
30000 registros produci